# ENVEDA CASMI 2026 — Colab Master Runner

This notebook is the **single Colab entry point** for the runtime repository:

`https://github.com/mybenkhadda/enveda-kaggle`

It does **not train or refit** the frozen model. It:

1. mounts Google Drive;
2. clones/updates the GitHub runtime repo;
3. installs Colab requirements;
4. detects the accelerator, preferring GPU when a validated backend exists;
5. copies the complete frozen bundle and competition files from Drive to local Colab storage;
6. validates the runtime repo and frozen bundle;
7. executes the Colab pipeline notebooks in a fresh kernel, in order;
8. saves executed notebooks, logs, `submission.csv`, `run_report.json`, and provenance back to Drive.

### Before running

In Colab choose **Runtime → Change runtime type → GPU**.

Expected Drive layout:

```text
MyDrive/EnvedaCASMI/
├── bundle/
├── competition/
│   ├── test.parquet
│   └── sample_submission.csv
└── results/
```

Future Colab notebooks should be named `NN_colab_*.ipynb` under `notebooks/`.  
If `notebooks/colab_pipeline.json` exists, that manifest controls the execution order instead.


In [ ]:
# Configuration
from pathlib import Path
import os, sys, json, shutil, hashlib, subprocess, time, platform, re

REPO_URL = "https://github.com/mybenkhadda/enveda-kaggle.git"
BRANCH = "main"

REPO_DIR = Path("/content/enveda-kaggle")
RUNTIME_ROOT = Path("/content/casmi_runtime")
INPUT_ROOT = RUNTIME_ROOT / "input"
LOCAL_BUNDLE = INPUT_ROOT / "bundle"
LOCAL_COMP = INPUT_ROOT / "competition"
OUTPUT_ROOT = RUNTIME_ROOT / "output"
EXECUTED_ROOT = RUNTIME_ROOT / "executed_notebooks"
LOG_ROOT = RUNTIME_ROOT / "logs"

DRIVE_ROOT = Path("/content/drive/MyDrive/EnvedaCASMI")
DRIVE_BUNDLE = DRIVE_ROOT / "bundle"
DRIVE_COMP = DRIVE_ROOT / "competition"
DRIVE_RESULTS = DRIVE_ROOT / "results"

DEVICE_PREFERENCE = "gpu"
REQUIRE_GPU = False
EXPECTED_MODEL = "V1_TL_1K_TESTSIM_STRICT"
EXPECTED_AGGREGATOR = "MOST_CONFIDENT_SPECTRUM"

# If False, the notebook refreshes the local bundle whenever the Drive manifest hash changes.
FORCE_RECOPY_BUNDLE = False

print("Configuration loaded.")


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

required_drive = [
    DRIVE_BUNDLE / "manifest.json",
    DRIVE_BUNDLE / "config.json",
    DRIVE_COMP / "test.parquet",
    DRIVE_COMP / "sample_submission.csv",
]
missing = [str(p) for p in required_drive if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required Drive inputs:\n" + "\n".join(missing))

DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print("Drive inputs found.")


## 2. Detect GPU / runtime

In [ ]:
def run_text(cmd):
    p = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return p.returncode, p.stdout

gpu_available = False
gpu_name = None
rc, smi = run_text(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
if rc == 0 and smi.strip():
    gpu_available = True
    gpu_name = smi.strip().splitlines()[0]

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("CPU count:", os.cpu_count())
print("GPU available:", gpu_available)
print("GPU:", gpu_name or "none")

if REQUIRE_GPU and not gpu_available:
    raise RuntimeError("REQUIRE_GPU=True but this Colab runtime has no GPU. Select Runtime > Change runtime type > GPU.")

os.environ["CASMI_DEVICE_PREFERENCE"] = DEVICE_PREFERENCE
os.environ["CASMI_REQUIRE_GPU"] = "1" if REQUIRE_GPU else "0"


## 3. Clone or update the GitHub runtime repo

In [ ]:
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Repo:", REPO_DIR)
print("Commit:", commit)


## 4. Install runtime requirements

In [ ]:
req_candidates = [
    REPO_DIR / "requirements-colab.txt",
    REPO_DIR / "colab" / "requirements-colab.txt",
]
req = next((p for p in req_candidates if p.exists()), None)

if req is None:
    raise FileNotFoundError(
        "requirements-colab.txt not found. Expected at repo root or colab/requirements-colab.txt"
    )

print("Installing:", req)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
print("Runtime requirements installed.")


## 5. Validate the lightweight GitHub repo

In [ ]:
repo_check_candidates = [
    REPO_DIR / "scripts" / "check_runtime_repo.py",
    REPO_DIR / "scripts" / "check_colab_repo.py",
]
repo_check = next((p for p in repo_check_candidates if p.exists()), None)

if repo_check is None:
    raise FileNotFoundError("Runtime repo checker not found under scripts/.")

subprocess.run(
    [sys.executable, str(repo_check), "--repo", str(REPO_DIR)],
    cwd=str(REPO_DIR),
    check=True,
)
print("Runtime repo check completed.")


## 6. Copy frozen bundle + competition inputs to local Colab SSD

In [ ]:
def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
EXECUTED_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

drive_manifest_hash = sha256_file(DRIVE_BUNDLE / "manifest.json")
marker = LOCAL_BUNDLE / ".drive_manifest_sha256"
reuse = (
    not FORCE_RECOPY_BUNDLE
    and LOCAL_BUNDLE.exists()
    and marker.exists()
    and marker.read_text().strip() == drive_manifest_hash
)

if reuse:
    print("Reusing local bundle copy:", LOCAL_BUNDLE)
else:
    if LOCAL_BUNDLE.exists():
        shutil.rmtree(LOCAL_BUNDLE)
    print("Copying complete bundle from Drive to local Colab storage...")
    shutil.copytree(DRIVE_BUNDLE, LOCAL_BUNDLE)
    marker.write_text(drive_manifest_hash)

LOCAL_COMP.mkdir(parents=True, exist_ok=True)
for name in ("test.parquet", "sample_submission.csv"):
    shutil.copy2(DRIVE_COMP / name, LOCAL_COMP / name)

required_local = [
    LOCAL_BUNDLE / "manifest.json",
    LOCAL_BUNDLE / "config.json",
    LOCAL_COMP / "test.parquet",
    LOCAL_COMP / "sample_submission.csv",
]
missing = [str(p) for p in required_local if not p.exists()]
if missing:
    raise FileNotFoundError("Local runtime copy incomplete:\n" + "\n".join(missing))

print("Bundle:", LOCAL_BUNDLE)
print("Competition:", LOCAL_COMP)
print("Output:", OUTPUT_ROOT)


## 7. Validate frozen bundle and expose runtime paths

In [ ]:
cfg = json.loads((LOCAL_BUNDLE / "config.json").read_text(encoding="utf-8"))

bundle_version = cfg.get("bundle_version")
model_id = cfg.get("model_id")
aggregator = cfg.get("aggregator_name") or cfg.get("aggregator")

if model_id != EXPECTED_MODEL:
    raise RuntimeError(f"Unexpected model_id: {model_id!r}")
if aggregator != EXPECTED_AGGREGATOR:
    raise RuntimeError(f"Unexpected aggregator: {aggregator!r}")

config_hash = cfg.get("CONFIG_HASH") or cfg.get("config_hash") or "unknown"

os.environ["CASMI_RUNTIME_ROOT"] = str(RUNTIME_ROOT)
os.environ["CASMI_INPUT_ROOT"] = str(INPUT_ROOT)
os.environ["CASMI_BUNDLE"] = str(LOCAL_BUNDLE)
os.environ["CASMI_COMPETITION"] = str(LOCAL_COMP)
os.environ["CASMI_OUTPUT"] = str(OUTPUT_ROOT)
os.environ["CASMI_KAGGLE_INPUT"] = str(INPUT_ROOT)
os.environ["CASMI_KAGGLE_WORKING"] = str(OUTPUT_ROOT)

print("bundle_version:", bundle_version)
print("model_id:", model_id)
print("aggregator:", aggregator)
print("CONFIG_HASH:", config_hash)


## 8. Run frozen bundle self-test (hard gate)

In [ ]:
bundle_check = REPO_DIR / "scripts" / "v63_bundle_check.py"
if not bundle_check.exists():
    raise FileNotFoundError(f"Missing bundle checker: {bundle_check}")

cmd = [
    sys.executable,
    str(bundle_check),
    "--bundle", str(LOCAL_BUNDLE),
    "--expect", EXPECTED_AGGREGATOR,
]

print("Running bundle self-test...")
p = subprocess.run(
    cmd,
    cwd=str(REPO_DIR),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(p.stdout)

if p.returncode != 0:
    raise RuntimeError("Bundle self-test FAILED. Inference is blocked.")

print("BUNDLE SELF-TEST PASS")


## 9. Resolve Colab pipeline notebooks

Preferred control file:

`notebooks/colab_pipeline.json`

Supported form:

```json
{
  "notebooks": [
    "notebooks/01_colab_inference.ipynb",
    "notebooks/03_colab_next_stage.ipynb"
  ]
}
```

If the manifest does not exist, the runner automatically executes every `NN_colab_*.ipynb` in lexical order, excluding this master notebook.


In [ ]:
manifest_path = REPO_DIR / "notebooks" / "colab_pipeline.json"

if manifest_path.exists():
    obj = json.loads(manifest_path.read_text(encoding="utf-8"))
    rel_notebooks = obj.get("notebooks", obj if isinstance(obj, list) else [])
    pipeline = [REPO_DIR / p for p in rel_notebooks]
else:
    nb_dir = REPO_DIR / "notebooks"
    if not nb_dir.exists():
        raise FileNotFoundError("Repository has no notebooks/ directory yet.")

    pipeline = sorted(
        p for p in nb_dir.glob("[0-9][0-9]_colab_*.ipynb")
        if p.name != "00_colab_master.ipynb"
    )

# Never recursively execute the master notebook.
pipeline = [p for p in pipeline if p.name != "00_colab_master.ipynb"]

missing = [str(p) for p in pipeline if not p.exists()]
if missing:
    raise FileNotFoundError("Pipeline notebook(s) missing:\n" + "\n".join(missing))

if not pipeline:
    raise RuntimeError(
        "No Colab pipeline notebooks found. Add notebooks/01_colab_inference.ipynb "
        "or notebooks/colab_pipeline.json."
    )

print("Execution plan:")
for i, p in enumerate(pipeline, 1):
    print(f"  {i}. {p.relative_to(REPO_DIR)}")


## 10. Execute inference and all configured future Colab notebooks

In [ ]:
def tail(path, n=80):
    try:
        lines = Path(path).read_text(errors="replace").splitlines()
        return "\n".join(lines[-n:])
    except Exception:
        return ""

pipeline_results = []
pipeline_start = time.time()

for idx, nb_path in enumerate(pipeline, 1):
    rel = nb_path.relative_to(REPO_DIR)
    executed = EXECUTED_ROOT / nb_path.name
    log_path = LOG_ROOT / f"{nb_path.stem}.log"

    print(f"\n[{idx}/{len(pipeline)}] Executing {rel}")
    t0 = time.time()

    cmd = [
        sys.executable, "-m", "jupyter", "nbconvert",
        "--to", "notebook",
        "--execute",
        "--ExecutePreprocessor.kernel_name=python3",
        "--ExecutePreprocessor.timeout=-1",
        "--output", str(executed),
        str(nb_path),
    ]

    env = os.environ.copy()
    # Let each child notebook see the repo orchestration code, but production
    # inference must still promote bundle/code/casmi_infer internally.
    env["PYTHONPATH"] = str(REPO_DIR / "src")

    with open(log_path, "w", encoding="utf-8") as log:
        proc = subprocess.run(
            cmd,
            cwd=str(REPO_DIR),
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )

    elapsed = time.time() - t0
    item = {
        "notebook": str(rel),
        "executed_notebook": str(executed),
        "log": str(log_path),
        "returncode": proc.returncode,
        "runtime_seconds": elapsed,
    }
    pipeline_results.append(item)

    print(f"Return code: {proc.returncode} | {elapsed/60:.1f} min")

    if proc.returncode != 0:
        print("\n--- LOG TAIL ---")
        print(tail(log_path))
        raise RuntimeError(f"Pipeline stopped because {rel} failed.")

print("\nAll configured Colab notebooks completed.")


## 11. Validate outputs and save everything back to Drive

In [ ]:
submission = OUTPUT_ROOT / "submission.csv"
run_report = OUTPUT_ROOT / "run_report.json"

if not submission.exists():
    raise FileNotFoundError(f"Expected inference output missing: {submission}")
if not run_report.exists():
    raise FileNotFoundError(f"Expected run report missing: {run_report}")

report = json.loads(run_report.read_text(encoding="utf-8"))

reported_model = report.get("model_id")
reported_agg = report.get("aggregator")
self_test_status = str(report.get("self_test_status", "")).upper()
submission_status = str(report.get("submission_validation_status", "")).upper()
infer_source = str(report.get("casmi_infer_source", ""))

if reported_model != EXPECTED_MODEL:
    raise RuntimeError(f"run_report model mismatch: {reported_model}")
if reported_agg != EXPECTED_AGGREGATOR:
    raise RuntimeError(f"run_report aggregator mismatch: {reported_agg}")
if self_test_status not in {"PASS", "PASSED", "TRUE"}:
    raise RuntimeError(f"run_report self-test not PASS: {self_test_status}")
if submission_status not in {"PASS", "PASSED", "TRUE"}:
    raise RuntimeError(f"submission validation not PASS: {submission_status}")

# If source is reported, it must be the frozen bundle implementation.
if infer_source:
    norm_source = infer_source.replace("\\", "/")
    norm_bundle = str(LOCAL_BUNDLE).replace("\\", "/")
    if norm_bundle not in norm_source:
        raise RuntimeError(
            "Inference did not report the frozen bundle code as its casmi_infer source:\n"
            + infer_source
        )

stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
safe_hash = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(config_hash))[:32]
result_dir = DRIVE_RESULTS / f"{stamp}_{safe_hash}"
result_dir.mkdir(parents=True, exist_ok=False)

# Core outputs
for src in (submission, run_report):
    shutil.copy2(src, result_dir / src.name)

# Logs and executed notebooks
shutil.copytree(LOG_ROOT, result_dir / "logs")
shutil.copytree(EXECUTED_ROOT, result_dir / "executed_notebooks")

provenance = {
    "created_at_utc": stamp,
    "repo_url": REPO_URL,
    "repo_commit": commit,
    "bundle_version": bundle_version,
    "CONFIG_HASH": config_hash,
    "model_id": EXPECTED_MODEL,
    "aggregator": EXPECTED_AGGREGATOR,
    "device_preference": DEVICE_PREFERENCE,
    "gpu_available": gpu_available,
    "gpu_name": gpu_name,
    "submission_sha256": sha256_file(submission),
    "run_report_sha256": sha256_file(run_report),
    "pipeline_runtime_seconds": time.time() - pipeline_start,
    "pipeline": pipeline_results,
}
(result_dir / "provenance.json").write_text(
    json.dumps(provenance, indent=2),
    encoding="utf-8",
)

print("RESULTS SAVED:", result_dir)
print("submission:", result_dir / "submission.csv")
print("run report:", result_dir / "run_report.json")


## 12. Final summary

In [ ]:
print("=" * 72)
print("CASMI COLAB PIPELINE COMPLETE")
print("=" * 72)
print("Repo commit :", commit)
print("Bundle      :", bundle_version)
print("CONFIG_HASH :", config_hash)
print("Model       :", EXPECTED_MODEL)
print("Aggregator  :", EXPECTED_AGGREGATOR)
print("GPU         :", gpu_name if gpu_available else "not available / CPU fallback")
print("Results     :", result_dir)
print()
print("Next:")
print("1. Inspect run_report.json.")
print("2. Confirm the actual backend and frozen casmi_infer source.")
print("3. Submit submission.csv manually to Kaggle only after validation is PASS.")
print("4. Add future Colab notebooks as NN_colab_*.ipynb or list them in notebooks/colab_pipeline.json.")
